In [0]:
# Databricks notebook: NASA POWER API ingestion into bronze.climatedata (multi-location, daily)

# ------------------------------
# IMPORTS
# ------------------------------
import requests
import pandas as pd
from datetime import datetime, timedelta
from delta.tables import DeltaTable
from pyspark.sql.functions import col
from pyspark.sql.utils import AnalysisException

# ------------------------------
# CONFIGURATION
# ------------------------------
BRONZE_TABLE = "eo_data.bronze.climate_nasa_pwr_api"

# Define multiple locations
locations = [
    {"country": "India", "location_name": "New Delhi", "latitude": 28.6, "longitude": 77.2},
    {"country": "India", "location_name": "Mumbai", "latitude": 19.07, "longitude": 72.87},
    {"country": "United Kingdom", "location_name": "London", "latitude": 51.52, "longitude": -0.11},
    {"country": "USA", "location_name": "Washington Park", "latitude": 46.6, "longitude": -120.49}
]

# Set NASA POWER API temporal range

# ------------------------------
# DYNAMIC DATE RANGE
# ------------------------------
# Fetch data for the previous day
END_DATE = datetime.now() - timedelta(days=1)  # yesterday
START_DATE = END_DATE - timedelta(days=6)       # 7-day window

START_DATE = START_DATE.strftime("%Y%m%d")
END_DATE = END_DATE.strftime("%Y%m%d")
# API base URL
API_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"

# Parameters to fetch
API_PARAMS = {
    "parameters": "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,WS10M,WS50M,WD10M,RH2M,"
                  "ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DWN,TOA_SW_DWN",
    "community": "SB",
    "format": "JSON",
    "start": START_DATE,
    "end": END_DATE
}

# ------------------------------
# FETCH DATA FOR ALL LOCATIONS
# ------------------------------
all_records = []

for loc in locations:
    params = API_PARAMS.copy()
    params.update({"latitude": loc["latitude"], "longitude": loc["longitude"]})
    
    try:
        response = requests.get(API_URL, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        if "properties" in data and "parameter" in data["properties"]:
            parameters = data["properties"]["parameter"]
            dates = list(parameters["T2M"].keys())
            
            for d in dates:
                row = {
                    "country": loc["country"],
                    "location_name": loc["location_name"],
                    "latitude": loc["latitude"],
                    "longitude": loc["longitude"],
                    "YEAR": int(d[0:4]),
                    "MO": int(d[4:6]),
                    "DY": int(d[6:8]),
                    "HR": 0,  # daily data
                    "T2M": parameters["T2M"].get(d),
                    "PRECTOTCORR": parameters["PRECTOTCORR"].get(d),
                    "WS10M": parameters["WS10M"].get(d),
                    "WS50M": parameters["WS50M"].get(d),
                    "WD10M": parameters["WD10M"].get(d),
                    "RH2M": parameters["RH2M"].get(d),
                    "ALLSKY_SFC_SW_DWN": parameters["ALLSKY_SFC_SW_DWN"].get(d),
                    "CLRSKY_SFC_SW_DWN": parameters["CLRSKY_SFC_SW_DWN"].get(d),
                    "TOA_SW_DWN": parameters["TOA_SW_DWN"].get(d)
                }
                all_records.append(row)
        else:
            print(f"No data returned for {loc['location_name']}")
            
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for {loc['location_name']}: {e}")

# ------------------------------
# CONVERT TO SPARK DATAFRAME
# ------------------------------
if all_records:
    pdf = pd.DataFrame(all_records)
    df = spark.createDataFrame(pdf)
    
    # Cast numeric columns
    numeric_cols = ["latitude","longitude","T2M","PRECTOTCORR","WS10M","WS50M","WD10M","RH2M",
                    "ALLSKY_SFC_SW_DWN","CLRSKY_SFC_SW_DWN","TOA_SW_DWN"]
    
    for col_name in numeric_cols:
        df = df.withColumn(col_name, col(col_name).cast("double"))

    # ------------------------------
    # UPSERT INTO BRONZE DELTA TABLE
    # ------------------------------
    try:
        spark.table(BRONZE_TABLE)
        table_exists = True
    except AnalysisException:
        table_exists = False

    if table_exists:
        delta_table = DeltaTable.forName(spark, BRONZE_TABLE)
        delta_table.alias("t").merge(
            source=df.alias("s"),
            condition="""t.location_name = s.location_name 
                        AND t.YEAR = s.YEAR 
                        AND t.MO = s.MO 
                        AND t.DY = s.DY"""
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    else:
        df.write.format("delta").option("mergeSchema", "true").saveAsTable(BRONZE_TABLE)

    print("Data ingestion/upsert completed successfully!")
else:
    print("No records to ingest.")


Data ingestion/upsert completed successfully!
